## Aaliyah John-Harry
### DATA 620 Project 3
#### 04/02/2026

In [1]:
import random
import nltk

from nltk.corpus import names
from nltk.classify import apply_features
from nltk.metrics import ConfusionMatrix
random.seed(2705)

In [2]:
# Data Loading
labeled_names = (
    [(name, "male") for name in names.words("male.txt")] +
    [(name, "female") for name in names.words("female.txt")]
)

random.shuffle(labeled_names)

## Introduction

For this project, I used the Names Corpus from the Natural Language Processing with Python to build and evaluate a machine learning classifier that predicts the gender of a name based on its linguistic features.

I began by loading and preprocessing the dataset consisting of labeled male and female names. The combined dataset was shuffled and split into three subsets: a training set, a dev-test set, and a test set. The training set was used to build the models, the dev-test set was used to evaluate and refine feature selection and model choice, and the test set was reserved for final performance evaluation.

Starting with a baseline classifier that uses only the last letter of each name as a feature, I incrementally improved the model by incorporating additional features such as prefixes, suffixes, name length, and vowel-based characteristics. These features were designed to capture patterns in how names are structured and how those patterns relate to gender.

I then trained and compared multiple classifiers, including Naive Bayes and Decision Tree models, using the dev-test set to guide model selection. After identifying the best-performing model, I evaluated its final performance on the test set.

Finally, I analyzed the model’s behavior through informative feature inspection, error analysis, and a confusion matrix to better understand its strengths, limitations, and overall effectiveness in classifying names by gender.

## Data Overview and Preprocessing

The dataset contains two classes:
- Male names: 2,944
- Female names: 5,000

To ensure randomness and avoid ordering bias, the combined dataset was shuffled before being split into three subsets:

- Training set: 6,944 names
- Dev-test set: 500 names
- Test set: 500 names

No extensive cleaning was required since the data consists of simple text strings. However, all names were converted to lowercase during feature extraction to ensure consistency. 

Feature engineering was then applied to transform each name into a set of attributes. The baseline model used only the final letter of each name, while the improved model incorporated additional features such as prefixes, suffixes (last 2–3 letters), name length, vowel count, and indicators for whether a name starts or ends with a vowel. These features allow the model to capture structural patterns in names that are predictive of gender.

In [3]:
test_names = labeled_names[:500]
devtest_names = labeled_names[500:1000]
train_names = labeled_names[1000:]

print("Train size:", len(train_names))
print("Dev-test size:", len(devtest_names))
print("Test size:", len(test_names))

Train size: 6944
Dev-test size: 500
Test size: 500


In [4]:
# Baseline feature extractor

def gender_features_baseline(name):
    name = name.lower()
    return {
        "last_letter": name[-1]
    }

In [5]:
# Improved feature extractor

def gender_features(name):
    name = name.lower()
    return {
        "first_letter": name[0],
        "last_letter": name[-1],
        "first2": name[:2],
        "last2": name[-2:],
        "last3": name[-3:],
        "name_length": len(name),
        "vowel_count": sum(1 for ch in name if ch in "aeiou"),
        "endswith_vowel": name[-1] in "aeiou",
        "startswith_vowel": name[0] in "aeiou",
    }

In [6]:
def evaluate_classifier(feature_fn, classifier_type="naive_bayes"):
    train_set = apply_features(feature_fn, train_names)
    devtest_set = apply_features(feature_fn, devtest_names)
    test_set = apply_features(feature_fn, test_names)

    if classifier_type == "naive_bayes":
        classifier = nltk.NaiveBayesClassifier.train(train_set)

    elif classifier_type == "decision_tree":
        classifier = nltk.DecisionTreeClassifier.train(
            train_set,
            binary=True,
            entropy_cutoff=0.01,
            depth_cutoff=20,
            support_cutoff=5
        )

    elif classifier_type == "maxent":
        classifier = nltk.MaxentClassifier.train(
            train_set,
            algorithm="iis",
            trace=0,
            max_iter=25
        )
    else:
        raise ValueError("classifier_type must be 'naive_bayes', 'decision_tree', or 'maxent'")

    devtest_accuracy = nltk.classify.accuracy(classifier, devtest_set)
    test_accuracy = nltk.classify.accuracy(classifier, test_set)

    return classifier, devtest_accuracy, test_accuracy, devtest_set, test_set


## Model Comparison and Selection

The baseline Naive Bayes classifier achieved a dev-test accuracy of 0.7660 and a test accuracy of 0.7780. This provides a solid starting point and is consistent with expectations when using only the final letter of a name as a feature.

After incorporating additional features such as prefixes, suffixes, name length, and vowel-related characteristics, both models showed improved performance. The Naive Bayes classifier improved to a dev-test accuracy of 0.7800 and a test accuracy of 0.8020, indicating that the added features provided meaningful information for classification.

The Decision Tree classifier performed slightly better, achieving a dev-test accuracy of 0.7880 and a test accuracy of 0.8260. Based on its higher dev-test accuracy, the Decision Tree was selected as the best-performing model.

Overall, the results show that adding more informative features improves classification performance and that the Decision Tree model is able to better capture patterns in the data compared to the baseline approach.

In [9]:
# baseline model
baseline_clf, baseline_dev_acc, baseline_test_acc, _, _ = evaluate_classifier(
    gender_features_baseline,
    classifier_type="naive_bayes"
)

print("Baseline Naive Bayes")
print("Dev-test accuracy:", round(baseline_dev_acc, 4))
print("Test accuracy:", round(baseline_test_acc, 4))


Baseline Naive Bayes
Dev-test accuracy: 0.766
Test accuracy: 0.778


In [10]:
# improved models
nb_clf, nb_dev_acc, nb_test_acc, nb_devset, nb_testset = evaluate_classifier(
    gender_features,
    classifier_type="naive_bayes"
)

dt_clf, dt_dev_acc, dt_test_acc, _, _ = evaluate_classifier(
    gender_features,
    classifier_type="decision_tree"
)

print("Baseline Naive Bayes")
print(f"Dev-test: {baseline_dev_acc:.4f} | Test: {baseline_test_acc:.4f}")

print("\nImproved Models")
print(f"Naive Bayes  -> Dev-test: {nb_dev_acc:.4f} | Test: {nb_test_acc:.4f}")
print(f"Decision Tree-> Dev-test: {dt_dev_acc:.4f} | Test: {dt_test_acc:.4f}")

Baseline Naive Bayes
Dev-test: 0.7660 | Test: 0.7780

Improved Models
Naive Bayes  -> Dev-test: 0.7800 | Test: 0.8020
Decision Tree-> Dev-test: 0.7880 | Test: 0.8260


In [11]:
if nb_dev_acc >= dt_dev_acc:
    best_name = "Naive Bayes"
    best_clf = nb_clf
    best_dev_acc = nb_dev_acc
    best_test_acc = nb_test_acc
else:
    best_name = "Decision Tree"
    best_clf = dt_clf
    best_dev_acc = dt_dev_acc
    best_test_acc = dt_test_acc

print("Best model based on dev-test accuracy:", best_name)
print(f"Best dev-test accuracy: {best_dev_acc:.4f}")
print(f"Best test accuracy: {best_test_acc:.4f}")


Best model based on dev-test accuracy: Decision Tree
Best dev-test accuracy: 0.7880
Best test accuracy: 0.8260


## Informative Features

Although the Decision Tree classifier performed best overall, the Naive Bayes classifier provides useful insight into which features are most predictive. The most informative features highlight strong patterns in name endings that are associated with gender.

For example, names ending in “na” or “la” are significantly more likely to be female, while names ending in consonants such as “k” or suffixes like “rd” and “us” are more likely to be male. These results show that suffix-based features are highly informative and play a major role in the model’s performance.

In [12]:
print("Most Informative Features (Naive Bayes):")
nb_clf.show_most_informative_features(15)

Most Informative Features (Naive Bayes):
Most Informative Features
                   last2 = 'na'           female : male   =     94.0 : 1.0
                   last2 = 'la'           female : male   =     73.6 : 1.0
             last_letter = 'k'              male : female =     43.9 : 1.0
                   last2 = 'ia'           female : male   =     37.8 : 1.0
                   last2 = 'sa'           female : male   =     34.7 : 1.0
             last_letter = 'a'            female : male   =     31.4 : 1.0
                   last2 = 'rd'             male : female =     30.3 : 1.0
                   last2 = 'rt'             male : female =     29.3 : 1.0
                   last2 = 'us'             male : female =     28.0 : 1.0
                   last3 = 'ard'            male : female =     26.8 : 1.0
                   last2 = 'io'             male : female =     25.8 : 1.0
                   last3 = 'ana'          female : male   =     24.9 : 1.0
                   last2 = 'ta'  

## Error Analysis 

I performed error analysis on the dev-test set to better understand where the model was making mistakes. Many of the misclassified names were either gender-ambiguous or did not follow typical naming patterns.

For example, names such as “Jessy,” “Gale,” “Quinn,” and “Emery” are commonly used for both males and females, making them difficult to classify based on simple character-based features. Additionally, some names in the dataset do not align with common expectations. For instance, “Meredeth” is labeled as male in the dataset but is more commonly associated with female names, which can lead to misclassification.

The model also struggled with shorter names like “Ty” and “Del,” since they provide fewer features for the classifier to use. Overall, many errors occur when names either lack strong suffix patterns or contradict the suffix patterns learned during training.

This analysis suggests that while suffix-based features are effective, they are not sufficient to perfectly classify ambiguous or uncommon names.

In [16]:
errors = []
for (name, true_label) in devtest_names:
    predicted = best_clf.classify(gender_features(name))
    if predicted != true_label:
        errors.append((true_label, predicted, name))

print("Sample dev-test errors:")
for item in errors[:25]:
    print(item)


Sample dev-test errors:
('male', 'female', 'Salomone')
('female', 'male', 'Sukey')
('female', 'male', 'Fawn')
('female', 'male', 'Ikey')
('male', 'female', 'Del')
('female', 'male', 'Gale')
('female', 'male', 'Mab')
('female', 'male', 'Biddy')
('female', 'male', 'Pet')
('male', 'female', 'Meredeth')
('female', 'male', 'Jessy')
('male', 'female', 'Emery')
('female', 'male', 'Gill')
('female', 'male', 'Glenn')
('female', 'male', 'Stoddard')
('female', 'male', 'Anett')
('male', 'female', 'Tedie')
('male', 'female', 'Arie')
('male', 'female', 'Ollie')
('female', 'male', 'Abagail')
('male', 'female', 'Jeffie')
('female', 'male', 'Quinn')
('male', 'female', 'Wendel')
('male', 'female', 'Bennet')
('female', 'male', 'Megan')


In [17]:
# Confusion matrix on test set
gold = [label for (_, label) in test_names]
predicted = [best_clf.classify(gender_features(name)) for (name, _) in test_names]

print("Confusion Matrix on Test Set:")
print(ConfusionMatrix(gold, predicted))

Confusion Matrix on Test Set:
       |   f     |
       |   e     |
       |   m   m |
       |   a   a |
       |   l   l |
       |   e   e |
-------+---------+
female |<283> 42 |
  male |  45<130>|
-------+---------+
(row = reference; col = test)



## Confusion Matrix

The confusion matrix shows that the classifier correctly identified 283 female names and 130 male names. It misclassified 42 female names as male and 45 male names as female, resulting in an overall accuracy of 0.826.

The errors are relatively balanced between the two classes, indicating that the model does not strongly favor predicting one gender over the other. Most misclassifications are likely due to ambiguous or uncommon names that do not follow typical naming patterns.

Overall, the confusion matrix confirms that the model performs well and generalizes consistently across both classes, which aligns with the earlier error analysis.

## Conclusion

This project set out to determine how well linguistic patterns in names can be used to predict gender. Through feature engineering and model comparison, the results show that relatively simple structural features (particularly suffixes) carry strong predictive power.

Building on the baseline approach, the addition of prefix, suffix, and vowel-based features led to consistent performance improvements. The Decision Tree model ultimately provided the best results, suggesting that it was better able to capture interactions between these features compared to the baseline and Naive Bayes models.

At the same time, the analysis highlights the limitations of relying solely on character-based features. Misclassifications were most common among ambiguous or non-standard names, reinforcing that not all names follow predictable linguistic patterns.

Overall, the findings demonstrate that while simple feature-based approaches can be highly effective, there are inherent limits to what can be achieved without additional context. This aligns with the broader goal of the project: not only to build an accurate classifier, but also to understand the patterns and constraints underlying the task.